In [1]:
using ChainRulesCore
using PyCall
using MarineHydro
using Base.Threads
using FiniteDifferences
using DifferentiationInterface 
import ForwardDiff 
import Zygote
using Test


In [2]:
# Hydrostatics
function hydrostatic_results(r1, r2, d1, d2)
    rho_w = 1000.0 # density of fluid [kg/m^3]
    g = 9.81 # acceleration due to gravity [m/s^2]
    K = rho_w * g * pi * r1^2 # hydrostatic stiffness [kg/s^2]
    M = rho_w * (pi * r1^2 * d1 + (pi/3) * (d2-d1) * (r1^2 + r1*r2 + r2^2)) # mass of body [kg]
    return [M, K]
end

# Radiation
function radiation_program(mesh, omega, dof) 
    A, B = calculate_radiation_forces(mesh,dof,omega)
    return [A, B]
end

# Diffraction
function diffraction_program(mesh, omega, dof) 
    F_D = DiffractionForce(mesh,omega,dof)
    F_FK = FroudeKrylovForce(mesh,omega,dof)
    F_ex = F_D + F_FK
    return [real(F_ex),imag(F_ex)]
end

# Everything
dof = [0.0,0.0,1.0]

function compute_all_hydrodynamic_coefficinets(mesh, x, omega)
    r1 = x[1]
    r2 = x[2]
    d1 = x[3]
    d2 = x[4]

    M, K = hydrostatic_results(r1, r2, d1, d2)
    A, B = radiation_program(mesh, omega, dof)
    F_ex_real, F_ex_imag = diffraction_program(mesh, omega, dof) 

    return [M, K, A, B, F_ex_real, F_ex_imag]
end

compute_all_hydrodynamic_coefficinets (generic function with 1 method)

In [3]:
x_vals = [0.88, 0.35, 0.16, 0.37]
mesh = wavebot_mesh(x_vals[1],x_vals[2],x_vals[3],x_vals[4],(3,3,3),10)
minimum(mesh.nfaces)


60

In [4]:

omega = 1.1
function compute_all(x)
    mesh = wavebot_mesh(x[1],x[2],x[3],x[4],(3,3,3),10)
    return compute_all_hydrodynamic_coefficinets(mesh, x, omega)
end
@timev all_hydro_coeffs = compute_all(x_vals)


  6.104033 seconds (26.11 M allocations: 1.137 GiB, 4.70% gc time, 88.12% compilation time)
elapsed time (ns):  6104033500
gc time (ns):       286597200
bytes allocated:    1220381928
pool allocs:        26094863
non-pool GC allocs: 11353
malloc() calls:     69
realloc() calls:    10
free() calls:       59
minor collections:  14
full collections:   0


6-element Vector{Float64}:
   654.2272453321136
 23866.25213272077
  1313.1623756864992
   275.45664787180937
 19777.721405401997
  -303.07376074977634

In [5]:
backend = AutoForwardDiff()
x_val = [0.88, 0.35, 0.16, 0.37]

# AD
@timev AD_grad = value_and_jacobian(compute_all, backend, x_val)
display(AD_grad)


([654.2272453321136, 23866.25213272077, 1313.1623756864992, 275.45664787180937, 19777.721405401997, -303.07376074977634], [1348.685726186098 347.46014748703107 1171.0810215031554 1261.7683294367805; 54241.482119819935 0.0 0.0 0.0; … ; 42842.69687644651 -280.765345007314 -2051.7320075787748 -499.7889085402494; -1311.7807706315307 7.660306499865905 60.105290961275585 14.393305012362804])

  9.508129 seconds (46.04 M allocations: 2.362 GiB, 4.45% gc time, 83.62% compilation time)
elapsed time (ns):  9508128600
gc time (ns):       423099500
bytes allocated:    2536312688
pool allocs:        46017486
non-pool GC allocs: 17648
malloc() calls:     137
realloc() calls:    17
free() calls:       122
minor collections:  23
full collections:   0


In [6]:

# FD
fdm = central_fdm(5, 1)
@timev FD_grad = FiniteDifferences.jacobian(fdm, compute_all, x_val)


 31.969290 seconds (723.40 M allocations: 20.129 GiB, 7.34% gc time, 2.25% compilation time)
elapsed time (ns):  31969290500
gc time (ns):       2346212300
bytes allocated:    21613860064
pool allocs:        723402693
non-pool GC allocs: 1056
malloc() calls:     385
free() calls:       6100
minor collections:  159
full collections:   0


([1348.685726187009 347.4601474877332 1171.0810215031968 1261.7683294363899; 54241.48211984789 4.6564891526758086e-9 4.4177699558381826e-9 4.2820428037782545e-9; … ; 42842.69687647505 -280.76534511837076 -2051.732007602213 -499.7889085821324; -1311.780770632035 7.660306500812918 60.105290962359504 14.393305015598512],)

In [7]:

@testset "Verify jacobian matrix with finite difference" begin
    @test FD_grad[1] ≈ AD_grad[2] atol=1e-6 rtol=1e-6
end

@testset "Verify value" begin
    @test all_hydro_coeffs ≈ AD_grad[1] atol=1e-6 rtol=1e-6
end

Test Summary:                                 | Pass  Total  Time
Verify jacobian matrix with finite difference |    1      1  0.5s
Test Summary: | Pass  Total  Time
Verify value  |    1      1  0.0s


Test.DefaultTestSet("Verify value", Any[], 1, false, false, true, 1.775577213285e9, 1.775577213315e9, false, "c:\\Users\\15183\\MarineHydro.jl\\dev\\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W6sZmlsZQ==.jl")

In [8]:
AD_grad[2]

6×4 Matrix{Float64}:
  1348.69   347.46      1171.08    1261.77
 54241.5      0.0          0.0        0.0
  4907.18  -308.008     -417.25    -715.215
  1193.05    -7.46511    -57.1328   -13.7522
 42842.7   -280.765    -2051.73    -499.789
 -1311.78     7.66031     60.1053    14.3933

In [9]:
FD_grad[1]

6×4 Matrix{Float64}:
  1348.69   347.46         1171.08        1261.77
 54241.5      4.65649e-9      4.41777e-9     4.28204e-9
  4907.18  -308.008        -417.25        -715.215
  1193.05    -7.46511       -57.1328       -13.7522
 42842.7   -280.765       -2051.73        -499.789
 -1311.78     7.66031        60.1053        14.3933

In [10]:


# Description of problem
omegas = [1.0, 1.5] # frequencies [rad/s]
beta = 0 # incident wave angle [rad]
t_DOFs = ["Heave"] # translational DOFs
r_DOFs = ["Pitch"] # rotational DOFs
DOFs = [t_DOFs; r_DOFs] # all DOFs

# Create Mesh object
radius = 1.5  
center = (0.0,0.0,0.0) 
len = 2.5
faces_max_radius = 0.7
cpt = pyimport("capytaine")
cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)

# Get MarineHydro values
mesh = Mesh(cptmesh)
rigid_dof_list = DOFs
rotation_center = [1.0, 1.0, 0.0] # off set for nonzero off-diagoinal elements
floatingbody = FloatingBody(mesh, rigid_dof_list, rotation_center, "Horizontal_Cylinder")

# Radiation solve functions
function A_and_B_vec(w)
    parameters = (wave_frequencies=w,
        radiating_dofs=collect(keys(floatingbody.dofs)),
        influenced_dofs=collect(keys(floatingbody.dofs)))
    data = compute_hydrodynamic_coefficients(parameters, floatingbody)
    return vcat(vec(data.added_mass), vec(data.radiation_damping)) 
end

# Incident + diffraction solve function
function F_ex_vec(w)
    parameters = (wave_frequencies=[w],
        wave_directions=[beta],
        influenced_dofs=collect(keys(floatingbody.dofs)))
    data = compute_hydrodynamic_coefficients(parameters, floatingbody)
    return vcat(real.(vec(data.excitation_force)),imag.(vec(data.excitation_force))) 
end

backend = AutoForwardDiff()



for omega in omegas
    @testset "Verify sensitivity of added mass and damping wrt Omega: $omega" begin
        A_and_B_FD = FiniteDifferences.central_fdm(5, 1)(A_and_B_vec, omega)
        A_and_B_AD = derivative(A_and_B_vec, backend, omega)
        @test A_and_B_AD !== nothing
        @test typeof(A_and_B_AD) == Vector{Float64}
        @test A_and_B_AD ≈ A_and_B_FD atol=1e-6 rtol=1e-6
    end
    @testset "Verify sensitivity of excitation force wrt Omega: $omega" begin
        F_ex_FD = FiniteDifferences.central_fdm(5, 1)(F_ex_vec, omega)
        F_ex_AD = derivative(F_ex_vec, backend, omega)
        @test F_ex_AD !== nothing
        @test typeof(F_ex_AD) == Vector{Float64}
        @test F_ex_AD ≈ F_ex_FD atol=1e-6 rtol=1e-6
    end
end  

Test Summary:                                               | Pass  Total   Time
Verify sensitivity of added mass and damping wrt Omega: 1.0 |    3      3  10.2s
Test Summary:                                         | Pass  Total  Time
Verify sensitivity of excitation force wrt Omega: 1.0 |    3      3  8.9s
Test Summary:                                               | Pass  Total  Time
Verify sensitivity of added mass and damping wrt Omega: 1.5 |    3      3  4.9s
Test Summary:                                         | Pass  Total  Time
Verify sensitivity of excitation force wrt Omega: 1.5 |    3      3  2.5s


In [11]:
using ForwardDiff

wavenumber = 1.2

x = [
    ForwardDiff.Dual(1.0, (1.0, 0.0, 0.0)), 
    ForwardDiff.Dual(2.0, (0.1, 0.0, 0.0)),
    ForwardDiff.Dual(3.0, (0.0, 0.5, 0.0))  
]

xi = [
    ForwardDiff.Dual(1.1, (1.0, 2.0, 3.0)),
    ForwardDiff.Dual(2.0, (0.0, 1.0, 0.0)),
    ForwardDiff.Dual(4.0, (0.0, 0.0, 1.0))
]



dis_squared_raw = (x[1] - xi[1])^2 + (x[2] - xi[2])^2
purt = 1e-18
if dis_squared_raw==0
    # must perturb before sqrt since sqrt(0) returns NaN dual number derivative
    dis = sqrt(dis_squared_raw + purt^2) - purt 
else
    dis = sqrt(dis_squared_raw)
end
r = wavenumber * dis
    
    # dis  = sqrt(dis_squared)
# if dis == eps(Float64)
#     r = wavenumber * (dis - eps(Float64)) # remove perturbation
# else
#     r = wavenumber * 



Dual{Nothing}(0.1200000000000001,0.0,2.4,3.5999999999999996)

In [12]:

# green_functions = (Rankine(), RankineReflected(), GFWu())


# integral_gradient(green_functions, element_i, element_j, wavenumber; with_respect_to_first_variable=!direct)